# V106 - Self-training with fold-local teachers

Pre-registered correction: v104 showed plus 0.021 but its teacher trained on all users including validation folds, so validation labels leaked into pseudo-labels and the gain is invalid. Here each fold gets its own teacher that never sees the fold, pseudo-labels differ per fold, and the student trains on fold-train plus pseudo test only. If the gain survives, self-training is real. If it collapses, v104 is retracted and the family closes. Same locks.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 3131
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v106")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Fold-local teachers, student retrain, gate

No validation label enters any teacher. Pseudo-labels are fold-specific by construction.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train, n_test = len(train), len(test_ids)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
X = np.hstack([W, A]).astype(np.float32)
X_tr, X_te = X[:n_train], X[n_train:]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_t, oof_s, fids = np.zeros_like(Y), np.zeros_like(Y), np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(X_tr, strata, fold_groups)):
    r = np.random.RandomState(SEED + fi)
    pseudo = np.zeros((n_test, len(MODS)))
    for b in range(BAGS):
        idx = tr[r.choice(len(tr), len(tr), replace=True)]
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(X_tr[idx], Y[idx, j])
            pseudo[:, j] += m.predict(X_te) / BAGS
    Xa = np.vstack([X_tr[tr], X_te])
    Ya = np.vstack([Y[tr], pseudo])
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(X_tr[tr], Y[tr, j])
        oof_t[va, j] = m.predict(X_tr[va])
        s = Ridge(alpha=1.0)
        s.fit(Xa, Ya[:, j])
        oof_s[va, j] = s.predict(X_tr[va])
    fids[va] = fi + 1
    print(f"fold {fi + 1}: teacher={ndcg5(Y[va], oof_t[va]).mean():.5f} student={ndcg5(Y[va], oof_s[va]).mean():.5f}", flush=True)
base = ndcg5(Y, oof_t)
d = ndcg5(Y, oof_s) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_s[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"teacher-only={base.mean():.5f} self-trained={ndcg5(Y, oof_s).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT honest self-training" if (lo > GATE_FLOOR and wf >= 0) else "RETRACT v104: gain was teacher leakage, family closed")

fold 1: teacher=0.63565 student=0.63522


fold 2: teacher=0.61158 student=0.60482


fold 3: teacher=0.66423 student=0.66542


fold 4: teacher=0.61207 student=0.61229


fold 5: teacher=0.62717 student=0.62702


teacher-only=0.62756 self-trained=0.62653 delta=-0.00103 CI95=[-0.00227,+0.00026] worst-fold=-0.00676
RETRACT v104: gain was teacher leakage, family closed
